<a href="https://colab.research.google.com/github/psalazarec/fundamento_programacion_ejercicio1-python-ia/blob/main/Ejercicio4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Configuración del Agente Conversacional con Gemini y Gradio

Primero, instalamos las librerías necesarias para LangChain, Gemini, Gradio y las herramientas de búsqueda/cálculo. Estas versiones son compatibles con Python 3.12.

In [ ]:
!pip install --upgrade --quiet "langchain>=0.2.0" "langchain-google-genai>=0.0.19" "google-generativeai" "gradio" "duckduckgo-search" "langchain_community"

In [8]:
python3 -m pip install --upgrade pip setuptools wheel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [ ]:
!python3 -m pip install --upgrade pip setuptools wheel

Importamos los módulos necesarios y configuramos la clave de la API de Gemini. Si aún no tienes una, créala en Google AI Studio y guárdala en los secretos de Colab bajo el nombre `GOOGLE_API_KEY`.

In [6]:
import os
import gradio as gr
from google.colab import userdata
import google.generativeai as genai

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.agents import AgentExecutor # Attempting to import AgentExecutor from langchain_core
from langchain.agents import create_react_agent # create_react_agent remains here
from langchain.tools import Tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools.python.tool import PythonREPLTool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory import ConversationBufferMemory

# Obtener la clave de API de los secretos de Colab
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Función para truncar texto para el historial
def truncate_text(text, max_length=100):
    if len(text) > max_length:
        return text[:max_length] + " ..."
    return text

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


ModuleNotFoundError: No module named 'langchain_google_genai'

Ahora definimos las herramientas que el agente podrá utilizar: búsqueda web con DuckDuckGo y una calculadora basada en Python.

In [ ]:
# Herramienta de búsqueda web
search = DuckDuckGoSearchRun()
web_search_tool = Tool(
    name="DuckDuckGo Search",
    func=search.run,
    description="useful for when you need to answer questions about current events or general knowledge"
)

# Herramienta de calculadora
calculator_tool = PythonREPLTool(
    name="Calculator",
    description="useful for when you need to perform mathematical calculations. Input should be a valid Python expression."
)

tools = [web_search_tool, calculator_tool]
print("Herramientas definidas: ", [tool.name for tool in tools])

NameError: name 'DuckDuckGoSearchRun' is not defined

Inicializamos el modelo de lenguaje (LLM), la memoria de conversación y el agente de LangChain. El agente utilizará el modelo `gemini-pro` y la memoria `ConversationBufferMemory` para recordar interacciones anteriores.

In [ ]:
# Inicializar el LLM
llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=0.7)

# Configurar la memoria de conversación para el agente
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# Definir el prompt para el agente
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful AI assistant. Answer questions as accurately as possible "
            "using the tools provided. If a specific request is made (search or calculate), "
            "prioritize using the corresponding tool. Remember previous conversation. "
            "If asked to search, use the 'DuckDuckGo Search' tool. If asked to calculate, "
            "use the 'Calculator' tool. Be concise and direct."
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

# Crear el agente
agent = create_react_agent(llm, tools, prompt)

# Crear el ejecutor del agente con memoria
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,
    handle_parsing_errors=True
)

print("Agente inicializado con Gemini y herramientas.")

NameError: name 'ChatGoogleGenerativeAI' is not defined

Finalmente, construimos la interfaz de usuario con Gradio. Tendrá tres pestañas: Conversación, Búsqueda y Cálculo, cada una con su propio campo de entrada, un botón de envío, un cuadro de texto para la respuesta actual y un historial de conversación donde las respuestas largas se truncarán.

In [ ]:
with gr.Blocks(title="Agente Conversacional Gemini") as demo:
    gr.Markdown("# Agente Conversacional con Gemini, LangChain y Herramientas")

    with gr.Tabs():
        with gr.TabItem("Conversación"):
            with gr.Column():
                conv_input = gr.Textbox(label="Tu mensaje", placeholder="Escribe tu pregunta o comentario aquí...")
                conv_submit_btn = gr.Button("Enviar")
                conv_output_current = gr.Textbox(label="Respuesta actual del agente", interactive=False, lines=5)
                conv_history_chatbot = gr.Chatbot([], label="Historial de conversación (respuestas truncadas)", height=300)

        with gr.TabItem("Búsqueda"):
            with gr.Column():
                search_input = gr.Textbox(label="¿Qué quieres buscar en la web?", placeholder="Ej: Noticias de hoy sobre IA")
                search_submit_btn = gr.Button("Buscar")
                search_output_current = gr.Textbox(label="Resultado de la búsqueda actual", interactive=False, lines=5)
                search_history_chatbot = gr.Chatbot([], label="Historial de búsquedas (respuestas truncadas)", height=300)

        with gr.TabItem("Cálculo"):
            with gr.Column():
                calc_input = gr.Textbox(label="Introduce una expresión matemática", placeholder="Ej: 5 * (10 + 2) / 3")
                calc_submit_btn = gr.Button("Calcular")
                calc_output_current = gr.Textbox(label="Resultado del cálculo actual", interactive=False, lines=5)
                calc_history_chatbot = gr.Chatbot([], label="Historial de cálculos (respuestas truncadas)", height=300)

    # Estados para mantener el historial completo de cada pestaña
    conv_full_history_state = gr.State([])
    search_full_history_state = gr.State([])
    calc_full_history_state = gr.State([])

    def process_conversation(user_message, full_history_state):
        if not user_message:
            return "", "", [], full_history_state

        response = agent_executor.invoke({"input": user_message})
        ai_response = response["output"]

        # Actualizar el historial completo
        full_history_state.append((user_message, ai_response))

        # Preparar historial truncado para visualización
        truncated_history_for_display = [(item[0], truncate_text(item[1])) for item in full_history_state]

        return "", ai_response, truncated_history_for_display, full_history_state

    def process_search(user_message, full_history_state):
        if not user_message:
            return "", "", [], full_history_state

        # Dirigir explícitamente al agente para usar la herramienta de búsqueda
        response = agent_executor.invoke({"input": f"Busca en la web sobre: {user_message}"})
        ai_response = response["output"]

        full_history_state.append((user_message, ai_response))
        truncated_history_for_display = [(item[0], truncate_text(item[1])) for item in full_history_state]

        return "", ai_response, truncated_history_for_display, full_history_state

    def process_calculation(user_message, full_history_state):
        if not user_message:
            return "", "", [], full_history_state

        # Dirigir explícitamente al agente para usar la herramienta de calculadora
        response = agent_executor.invoke({"input": f"Calcula la siguiente expresión: {user_message}"})
        ai_response = response["output"]

        full_history_state.append((user_message, ai_response))
        truncated_history_for_display = [(item[0], truncate_text(item[1])) for item in full_history_state]

        return "", ai_response, truncated_history_for_display, full_history_state

    # Enlazar funciones a los eventos de los botones
    conv_submit_btn.click(
        process_conversation,
        inputs=[conv_input, conv_full_history_state],
        outputs=[conv_input, conv_output_current, conv_history_chatbot, conv_full_history_state]
    )
    search_submit_btn.click(
        process_search,
        inputs=[search_input, search_full_history_state],
        outputs=[search_input, search_output_current, search_history_chatbot, search_full_history_state]
    )
    calc_submit_btn.click(
        process_calculation,
        inputs=[calc_input, calc_full_history_state],
        outputs=[calc_input, calc_output_current, calc_history_chatbot, calc_full_history_state]
    )

    demo.launch(debug=True, share=True)

/tmp/ipykernel_16623/933868938.py:10: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  conv_history_chatbot = gr.Chatbot([], label="Historial de conversación (respuestas truncadas)", height=300)
/tmp/ipykernel_16623/933868938.py:10: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  conv_history_chatbot = gr.Chatbot([], label="Historial de conversación (respuestas truncadas)", height=300)
/tmp/ipykernel_16623/933868938.py:17: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated 

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://dfac5ea95c7727baac.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://dfac5ea95c7727baac.gradio.live


# Ejercicio 4 — Proyecto Final Integrador

**Módulo: Python para IA** | Máster en Inteligencia Artificial

**Tipo**: Proyecto final | **Sesión**: 4

---

## 🎯 Objetivo

Construir una **aplicación web con IA** y desplegarla online. Este proyecto integra todo lo aprendido en el módulo: Python, prompting, modelos de IA, interfaces y deployment.

No se evalúa la complejidad del código (podéis usar IA para generarlo), sino la **idea, la funcionalidad y la capacidad de llevarlo a producción**.

---
## ¿Qué tienes que construir?

Elige **una** de las tres opciones siguientes. Elige la que más te interese, no la más fácil — la motivación importa.

Todas las opciones requieren:
- Una **interfaz web** (Gradio o Streamlit)
- Un **modelo de IA** integrado (API de Gemini, HuggingFace, LangChain, etc.)
- **Deployment** online (HuggingFace Spaces o Streamlit Cloud)
- Un **repositorio GitHub** con el código

---
## Opción A — App Multi-IA con Gradio/Streamlit

Crear una aplicación que use IA generativa para **al menos 2 operaciones distintas**.

**Ejemplos de combinaciones:**
- Resumen + Traducción + Análisis de sentimiento
- Generación de texto + Clasificación + QA
- Extracción de entidades + Generación de respuestas

**¿Qué se espera?** Una interfaz donde el usuario pueda elegir entre diferentes capacidades de IA y ver los resultados. Puede organizarse con pestañas (tabs) o como una interfaz unificada.

> 🤖 **Prompt sugerido**: *"Crea una app Gradio con tabs que tenga: (1) un analizador de sentimiento con HuggingFace, (2) un resumidor de textos con Gemini, y (3) una herramienta de traducción. Cada tab tiene su propio input y output."*

---
## Opción B — Sistema RAG con Interfaz

Crear un **sistema de pregunta-respuesta sobre documentos propios**.

**Requisitos mínimos:**
- Al menos **10 documentos/fragmentos** indexados (sobre un tema que te interese)
- Al menos **3 preguntas de demo** que funcionen correctamente
- Interfaz con Gradio o Streamlit

**¿Qué se espera?** Un chatbot que responda preguntas basándose en tus documentos, no en conocimiento general. Ideal si quieres profundizar en lo aprendido en la Sesión 3 (LangChain + RAG).

> 🤖 **Prompt sugerido**: *"Crea una app Gradio con sistema RAG usando LangChain y Gemini. Los documentos son [describe tu tema]. Incluye un ChatInterface que muestre las fuentes usadas en cada respuesta."*

---
## Opción C — Agente Conversacional

Crear un **chatbot avanzado** con capacidades adicionales.

**Requisitos mínimos:**
- **Memoria**: recuerda lo dicho en la conversación
- Al menos **1 herramienta** integrada (búsqueda web, cálculo, consulta a API externa, etc.)
- Interfaz con Gradio o Streamlit

**¿Qué se espera?** Un chatbot que no solo responda preguntas, sino que pueda **hacer cosas**: buscar información actualizada, calcular, consultar datos en tiempo real. Es la opción más avanzada, pero también la más impresionante.

> 🤖 **Prompt sugerido**: *"Crea un agente con LangChain y Gemini que tenga memoria de conversación y herramientas de búsqueda web (DuckDuckGo) y calculadora. Despliégalo con Gradio ChatInterface."*

---
## ¿Cómo abordar el proyecto?

1. **Elige la opción** que más te interese y define el tema concreto de tu app.
2. **Escribe una spec**: qué hace la app, qué tecnologías usa, qué flujo sigue el usuario. Una spec clara permite pedirle a la IA que genere el código base completo.
3. **Pide a Gemini/Copilot** que genere el código a partir de la spec.
4. **Prueba en Colab o local** — haz que funcione lo básico antes de añadir más.
5. **Despliega pronto** (HuggingFace Spaces o Streamlit Cloud). No esperes al último día: un deploy inicial con la versión mínima te evita sorpresas.
6. **Itera**: prueba → mejora → vuelve a desplegar.

> 💡 **Empieza simple**: un prototipo funcional en 30 minutos vale más que un plan ambicioso que no se termina.

---
## Criterios de Evaluación

| Criterio | Peso | Descripción | Qué se valora |
|----------|:----:|-------------|---------------|
| **Funcionalidad** | 40% | La app funciona correctamente, sin errores | Que haga lo que promete. Que no se rompa con inputs inesperados. Que el flujo sea completo. |
| **Uso de IA generativa** | 25% | Integración efectiva de modelos/APIs | Que la IA aporte valor real. Uso correcto de prompts, parámetros y manejo de errores de API. |
| **UI/UX** | 15% | Interfaz clara, intuitiva y usable | Títulos claros, inputs con labels descriptivos, ejemplos de uso, feedback al usuario. |
| **Documentación** | 10% | README claro, type hints, docstrings | README con descripción, instrucciones de instalación y capturas de pantalla. |
| **Deployment** | 10% | App desplegada y accesible online | La URL funciona. Los secretos están bien gestionados (no hay API keys en el código). |

---
## Entrega

Sube en el formulario de **Blackboard** un archivo **docx** con:

- **Link al repositorio de GitHub** (debe ser público)
- **Link a la app desplegada** (HuggingFace Spaces o Streamlit Cloud)
- **Capturas de pantalla** de la app funcionando
- **Capturas del código** (las partes más relevantes)

**Archivos mínimos en el repositorio:**
```
mi-app/
├── app.py              # Código de la aplicación
├── requirements.txt    # Dependencias
└── README.md           # Descripción, instrucciones y link a la app
```

**El README.md debe incluir:**
- Descripción del proyecto (qué hace y por qué)
- Tecnologías utilizadas (Gradio/Streamlit, modelos, APIs)
- Instrucciones de instalación y ejecución local
- Link a la app desplegada
- Capturas de pantalla de la app funcionando

> ⚠️ Asegúrate de que ambos links son públicos y accesibles antes de entregar. Si la URL no funciona, la nota máxima será **30%** (solo se evalúa el código del repo).

---
## 💡 Tips

### Proceso recomendado

- **Usa prompting agresivamente**: este módulo se llama "Python para IA" — se espera que uses IA para desarrollar. Dale tu spec a Gemini y pídele que genere el proyecto completo. Itera sobre el resultado.
- **Git desde el inicio**: crea el repo de GitHub antes de empezar a codificar. Haz commits frecuentes. Así nunca pierdes trabajo y el historial documenta tu proceso.
- **Testa el deployment pronto**: haz un deploy inicial con la versión más simple posible e id actualizando. No esperes a tenerlo perfecto.

### Errores comunes a evitar

| Error | Consecuencia | Solución |
|-------|-------------|----------|
| API key en el código | Seguridad comprometida, repo bloqueado | Usar secrets de HF/Streamlit + `os.environ` |
| No testear antes del deploy | "En mi notebook funcionaba" | Probar en un entorno limpio |
| README vacío | El evaluador no sabe qué hace la app | Escribir el README antes del código |
| Ignorar errores de la API | La app se rompe cuando la API falla | `try/except` en todas las llamadas a APIs |
| Proyecto demasiado ambicioso | No se termina a tiempo | Empezar con lo mínimo funcional |

---
## 📚 Recursos

- [Gradio — Documentación](https://gradio.app/docs/)
- [Gradio — ChatInterface](https://www.gradio.app/docs/gradio/chatinterface)
- [Streamlit — Documentación](https://docs.streamlit.io/)
- [Streamlit — Cheat Sheet](https://docs.streamlit.io/develop/quick-reference/cheat-sheet)
- [HuggingFace Spaces — Crear un Space](https://huggingface.co/docs/hub/spaces-overview)
- [HuggingFace Spaces — Secrets](https://huggingface.co/docs/hub/spaces-overview#managing-secrets)
- [Streamlit Cloud — Deployment](https://docs.streamlit.io/deploy/streamlit-community-cloud)
- [Google AI — Python SDK](https://ai.google.dev/gemini-api/docs/quickstart?lang=python)

In [ ]:
# Comprueba tus links antes de entregar
repo_url = ""   # Ej: "https://github.com/tu-usuario/tu-app"
app_url  = ""   # Ej: "https://tu-usuario-tu-app.hf.space"

print(f"Repositorio:   {repo_url}")
print(f"App desplegada: {app_url}")

assert repo_url.startswith("https://github.com/"), "❌ El link del repo no parece correcto"
assert app_url.startswith("https://"), "❌ El link de la app no parece correcto"
print("✅ Links con formato correcto — recuerda comprobar que son públicos y accesibles")